In [0]:
import time
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql import types as T

dbutils.widgets.text("catalogo", "workspace")
dbutils.widgets.text("data_referencia_calculo", "2026-05-22")

catalogo = dbutils.widgets.get("catalogo").strip()
data_referencia_param = dbutils.widgets.get("data_referencia_calculo").strip()

spark.sql(f"USE CATALOG `{catalogo}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalogo}`.`gold`")

DATA_REFERENCIA_COL = F.to_date(F.lit(data_referencia_param))

TABELA_CLIENTES = f"{catalogo}.silver.tb_clientes"
TABELA_PEDIDOS = f"{catalogo}.silver.fat_pedidos"
TABELA_TICKETS = f"{catalogo}.silver.tb_tickets"
TABELA_CLICKSTREAM = f"{catalogo}.silver.tb_clickstream"
TABELA_AVALIACOES = f"{catalogo}.silver.tb_avaliacoes"

TABELA_DESTINO_CLIENTES = f"{catalogo}.gold.gold_cliente_360"

print(f"Catálogo em uso: {catalogo}")
print(f"Data de referência: {data_referencia_param}")
print(f"Tabela destino: {TABELA_DESTINO_CLIENTES}")

In [0]:
df_clientes = spark.table(TABELA_CLIENTES)
df_pedidos = spark.table(TABELA_PEDIDOS)
df_tickets = spark.table(TABELA_TICKETS)
df_clickstream = spark.table(TABELA_CLICKSTREAM)
df_avaliacoes = spark.table(TABELA_AVALIACOES)

df_avaliacoes_base_gold = (
    df_avaliacoes
    .join(
        F.broadcast(df_pedidos.select("id_pedido", "data_pedido").dropDuplicates(["id_pedido"])),
        on="id_pedido",
        how="inner"
    ).filter(F.col("data_avaliacao") >= F.col("data_pedido")).drop("data_pedido")
)

df_pedidos_validos_para_tickets = (
    df_pedidos
    .select(
        "id_pedido",
        F.col("id_cliente").alias("id_cliente_pedido")
    )
    .dropDuplicates(["id_pedido"])
)

df_tickets_base_gold = (
    df_tickets
    .join(
        F.broadcast(df_pedidos_validos_para_tickets),
        on="id_pedido",
        how="inner"
    )
    .drop("id_cliente")
    .withColumnRenamed("id_cliente_pedido", "id_cliente")
    .withColumn(
        "status_ticket",
        F.when(F.col("data_resolucao").isNull(), F.lit("Aberto"))
         .otherwise(F.lit("Resolvido"))
    )
)

In [0]:
df_clientes_gold = df_clientes.select(
    "id_cliente",
    F.concat_ws(
        " ",
        F.col("nome"),
        F.col("sobrenome")
    ).alias("nome"),

    "email",
    "telefone",
    "data_cadastro",
    "cidade",
    "estado",

    "origem",
    F.when(F.col("idade") >= 18, True).otherwise(False).alias("maior_de_idade")
)

In [0]:
df_pedidos_agg = df_pedidos.groupBy("id_cliente").agg(

    # quantidade total pedidos
    F.count("id_pedido").alias("qtd_pedidos_total"),

    # aprovados
    F.sum(
        F.when(
            F.col("status") == "Aprovado",
            1
        ).otherwise(0)
    ).alias("qtd_pedidos_aprovados"),

    # recusados
    F.sum(
        F.when(
            F.col("status") == "Recusado",
            1
        ).otherwise(0)
    ).alias("qtd_pedidos_recusados"),

    # reembolsados
    F.sum(
        F.when(
            F.col("status") == "Reembolsado",
            1
        ).otherwise(0)
    ).alias("qtd_pedidos_reembolsados"),

    # processando
    F.sum(
        F.when(
            F.col("status") == "Processando",
            1
        ).otherwise(0)
    ).alias("qtd_pedidos_processando"),

    # valor total gasto em pedidos aprovados
    F.round(
        F.sum(
            F.when(F.col("status") == "Aprovado", F.col("valor_total"))
             .otherwise(F.lit(0))
        ),
        2
    ).alias("valor_total_gasto"),

    # primeiro pedido
    F.min("data_pedido").alias("data_primeiro_pedido"),

    # último pedido
    F.max("data_pedido").alias("data_ultimo_pedido")
)

df_pedidos_agg = df_pedidos_agg.withColumn(
    "ticket_medio",
    F.when(
        F.col("qtd_pedidos_aprovados") > 0,
        F.round(F.col("valor_total_gasto") / F.col("qtd_pedidos_aprovados"), 2)
    ).otherwise(F.lit(None).cast("decimal(18,2)"))
)

In [0]:
df_tickets_agg = (
    df_tickets_base_gold
    .groupBy("id_cliente")
    .agg(
        F.count("*").alias("qtd_tickets_total"),

        F.sum(
            F.when(F.col("status_ticket") == "Aberto", F.lit(1))
             .otherwise(F.lit(0))
        ).alias("qtd_tickets_abertos"),

        F.sum(
            F.when(F.col("status_ticket") == "Resolvido", F.lit(1))
             .otherwise(F.lit(0))
        ).alias("qtd_tickets_resolvidos")
    )
)

In [0]:
df_avaliacoes.printSchema()

In [0]:
df_avaliacoes_agg = df_avaliacoes_base_gold.groupBy("id_cliente").agg(

    F.count("id_avaliacao").alias("qtd_avaliacoes"),

    F.round(
        F.avg("nota_produto"),
        2
    ).alias("nota_media_dada"),

    F.round(
        F.avg("nota_nps"),
        2
    ).alias("nps_medio_avaliacoes_cliente")
)

display(df_avaliacoes_agg)

In [0]:
df_clickstream = df_clickstream.groupBy("id_cliente").agg(

    F.count("*").alias("qtd_eventos_clickstream"),

    F.mode("canal").alias("canal_preferido")
)

In [0]:
df_gold = (
    df_clientes_gold

    .join(df_pedidos_agg, on="id_cliente", how="left")

    .join(df_tickets_agg, on="id_cliente", how="left")

    .join(df_avaliacoes_agg, on="id_cliente", how="left")

    .join(df_clickstream, on="id_cliente", how="left")
)

In [0]:
df_gold = df_gold.fillna({

    # pedidos
    "qtd_pedidos_total": 0,
    "qtd_pedidos_aprovados": 0,
    "qtd_pedidos_recusados": 0,
    "qtd_pedidos_reembolsados": 0,
    "qtd_pedidos_processando": 0,
    "valor_total_gasto": 0,

    # tickets
    "qtd_tickets_total": 0,
    "qtd_tickets_abertos": 0,
    "qtd_tickets_resolvidos": 0,

    # avaliações
    "qtd_avaliacoes": 0,

    # clickstream
    "qtd_eventos_clickstream": 0
})

In [0]:
df_gold = df_gold.withColumn(
    "segmento_ltv",

    F.when(F.col("valor_total_gasto") >= 5000, "Alto")

    .when(F.col("valor_total_gasto") >= 1000, "Medio")

    .otherwise("Baixo")
)

In [0]:
df_gold = df_gold.withColumn(
    "is_ativo_90d",
    
    F.when(
        F.col("data_ultimo_pedido").isNotNull(),
        F.datediff(DATA_REFERENCIA_COL, F.col("data_ultimo_pedido")) <= 90
    ).otherwise(F.lit(False))
)

In [0]:
df_gold = df_gold.withColumn(
    "is_em_risco",

    F.coalesce(
        F.col("qtd_tickets_abertos"),
        F.lit(0)
    ) >= 3
)

In [0]:
df_gold_cliente_360 = df_gold.withColumn(
    "data_referencia_calculo",
    DATA_REFERENCIA_COL
)

In [0]:

df_gold_cliente_360.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA_DESTINO_CLIENTES)

print(f"Tabela cliente 360 salva com sucesso em: {TABELA_DESTINO_CLIENTES}")

display(df_gold_cliente_360.limit(5))